# CEC 2026: Modified GUESS Approach Using Machine Decision- makers for an Efficient Interactive Multi-criterion Decision-Making Procedure

### Authors: Deepanshu Yadav, Kalyanmoy Deb

# Testing Machine-DM using Pre-trained ANNs

## Import Following Libraries

In [ ]:
from tensorflow import keras
import joblib
import numpy as np
import os

## Set directory of pretrained ANNs

In [2]:
user_dir = r"file path"
os.makedirs(user_dir, exist_ok=True)

## Call pretrained ANN1 for a specific problem (Crash worthiness here)

In [3]:
model_path = os.path.join(user_dir, "Ex3_ANN1.keras")
model1 = keras.models.load_model(model_path, compile=False)
model1.summary()

Model: "model_7"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_8 (InputLayer)           [(None, 5)]          0           []                               
                                                                                                  
 dense_28 (Dense)               (None, 128)          768         ['input_8[0][0]']                
                                                                                                  
 dense_29 (Dense)               (None, 64)           8256        ['dense_28[0][0]']               
                                                                                                  
 dense_30 (Dense)               (None, 32)           2080        ['dense_29[0][0]']               
                                                                                            

## Call pretrained ANN2 for a specific problem (Crash worthiness here)

In [4]:
model_path2 = os.path.join(user_dir, "Ex3_ANN2.keras")
model2 = keras.models.load_model(model_path2, compile=False)

scaler_X_path = os.path.join(user_dir, "Ex3_scaler_X.pkl")
scaler_X = joblib.load(scaler_X_path)

scaler_Y_path = os.path.join(user_dir, "Ex3_scaler_Y.pkl")
scaler_Y = joblib.load(scaler_Y_path)

model2.summary()

Model: "model_8"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_9 (InputLayer)           [(None, 5)]          0           []                               
                                                                                                  
 dense_32 (Dense)               (None, 128)          768         ['input_9[0][0]']                
                                                                                                  
 dense_33 (Dense)               (None, 64)           8256        ['dense_32[0][0]']               
                                                                                                  
 dense_34 (Dense)               (None, 32)           2080        ['dense_33[0][0]']               
                                                                                            

## Function: Given PO solution $X$, predicts $I$ and $[\bar{z},$ $\varepsilon]$ Using ANN1 and ANN2

In [5]:
def Output_MachDM_Pref(ann_model1, ann_model2, scaler_X, scaler_Y, x):
    probs = ann_model1.predict(x)
    pred_class = np.zeros(len(probs))
    for i in range(0,len(probs)):
        pred_class[i] = np.argmax(probs[i])+1
        
    X_scaled = scaler_X.transform(x)
    Y_new_scaled_list = ann_model2.predict(X_scaled, verbose=0)  
    Y_new_scaled = np.concatenate(Y_new_scaled_list, axis=1)
    Y_new_scaled_trimmed = Y_new_scaled[:, :scaler_Y.mean_.shape[0]]  
    pred_params = scaler_Y.inverse_transform(Y_new_scaled_trimmed)
    for i in range(0,len(pred_class)):
        if pred_class[i] ==1 or pred_class[i] ==3:
            pred_params[0,i] = 0
            
    return pred_class, pred_params

## A Sample Example

In [6]:
X_solns = np.array([[0.5,0.3,0.0,0.0,0.0]])
pred_class, pred_params = Output_MachDM_Pref(model1, model2, scaler_X, scaler_Y, X_solns)
print("Predicted Objective Class Vector from ANN1:", pred_class)
print("Predicted Bounding Parameter Vector from ANN2:", pred_params)

1/1 [==============================] - 0s 196ms/step
Predicted Objective Class Vector from ANN1: [4. 2. 3.]
Predicted Bounding Parameter Vector from ANN2: [[1659.7571      7.694066    0.      ]]
